# Stage-wise sentencing model analysis

This notebook reads verified sentencing annotations, creates a reproducible 80/20 judgment-level split, learns interpretable stage effects from the training partition only, and evaluates the derived sentence path on the held-out partition. It never writes to MongoDB. See [`../STAGE_MODEL_ANALYSIS_PLAN.md`](../STAGE_MODEL_ANALYSIS_PLAN.md) and [`../CURRENT_SYSTEM_DATA_SCHEMA.md`](../CURRENT_SYSTEM_DATA_SCHEMA.md) for the agreed method and source schema.

In [9]:
from __future__ import annotations

import json
import os
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from pymongo import MongoClient
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, median_absolute_error
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, SplineTransformer, StandardScaler

RANDOM_SEED = 42
TEST_SIZE = 0.20
MIN_FACTOR_SUPPORT = 10
BOOTSTRAP_ITERATIONS = 1_000
INFERRED_ROLE_SOURCE = 'Inferred as starting point since role adjustment not provided'
NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / 'stage_model_analysis.ipynb').exists():
    NOTEBOOK_DIR = Path.cwd().resolve() / 'notebooks'
CACHE_DIR = NOTEBOOK_DIR / '.cache'
CACHE_PATH = CACHE_DIR / 'stage_model_analysis_verified_features.json'
CACHE_METADATA_PATH = CACHE_DIR / 'stage_model_analysis_verified_features.metadata.json'
REFRESH_CACHE = False
OUTPUT_PATH = NOTEBOOK_DIR / 'stage_model_analysis.xlsx'

repo_root = Path.cwd().resolve()
if not (repo_root / 'featureExtraction').exists():
    repo_root = repo_root.parent

for env_path in (
    repo_root / 'featureExtraction' / '.env',
    repo_root / 'featureVerification' / '.env.local',
    repo_root / '.env',
):
    if env_path.exists():
        load_dotenv(env_path)

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 140)

## Load and flatten verified trials

The raw annotation payload stays intact in the flattened rows. Canonical model labels are added alongside the original factor labels; they never replace data in MongoDB. The initial pull is cached under `notebooks/.cache/`; subsequent runs use that cache unless `REFRESH_CACHE` is set to `True`.

In [10]:
CANONICAL_FACTOR_MAP = {
    'Import': 'Cross-border trafficking',
    'Export': 'Cross-border trafficking',
    'Refugee/Asylum': 'Refugee claimant',
}


def canonical_factor(name: str | None) -> str | None:
    if not name:
        return None
    return CANONICAL_FACTOR_MAP.get(name, name)


def unique_canonical_factors(factors: list[dict[str, Any]] | None) -> list[str]:
    return list(dict.fromkeys(
        factor_name
        for factor_name in (canonical_factor(item.get('factor')) for item in (factors or []))
        if factor_name
    ))


def total_months(detail: dict[str, Any] | None) -> float | None:
    if not detail:
        return None
    if detail.get('total_months') is not None:
        return float(detail['total_months'])
    years = detail.get('sentence_years')
    months = detail.get('sentence_months')
    if years is None and months is None:
        return None
    return float(years or 0) * 12 + float(months or 0)


def is_inferred(detail: dict[str, Any] | None) -> bool:
    if not detail:
        return False
    return bool(detail.get('inferred')) or detail.get('source') == INFERRED_ROLE_SOURCE


def direct_plea_reduction(plea: dict[str, Any], incoming_months: float | None) -> float | None:
    if not plea.get('pleaded_guilty') or is_inferred(plea):
        return None
    years = plea.get('reduction_years')
    months = plea.get('reduction_months')
    if years is not None or months is not None:
        return float(years or 0) * 12 + float(months or 0)
    percentage = plea.get('reduction_percentage')
    if percentage is not None and incoming_months is not None:
        return incoming_months * float(percentage) / 100
    return None


query = {'is_verified': True, 'exclude': {'$ne': True}}
projection = {
    'source_judgement_id': 1,
    'filename': 1,
    'judgement.neutral_citation': 1,
    'trials': 1,
}
if CACHE_PATH.exists() and not REFRESH_CACHE:
    documents = json.loads(CACHE_PATH.read_text())
    cache_metadata = json.loads(CACHE_METADATA_PATH.read_text()) if CACHE_METADATA_PATH.exists() else {}
    print(f'Loaded {len(documents):,} verified, non-excluded judgments from cache: {CACHE_PATH}')
    if cache_metadata:
        print(f"Cache created: {cache_metadata.get('created_at', 'unknown')}")
else:
    mongo_uri = os.getenv('DB_MONGODB_URI')
    if not mongo_uri:
        raise RuntimeError('DB_MONGODB_URI is required when the cache is missing or REFRESH_CACHE is True')
    client = MongoClient(mongo_uri)
    database = client.get_database(os.getenv('DB_NAME', 'drug-sentencing-predictor'))
    verified_collection = database.get_collection('verified-features')
    documents = list(verified_collection.find(query, projection))
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    temporary_cache_path = CACHE_PATH.with_suffix('.tmp')
    temporary_cache_path.write_text(json.dumps(documents, default=str))
    temporary_cache_path.replace(CACHE_PATH)
    cache_metadata = {
        'created_at': pd.Timestamp.now(tz='UTC').isoformat(),
        'document_count': len(documents),
        'query': query,
        'projection_fields': sorted(projection),
    }
    CACHE_METADATA_PATH.write_text(json.dumps(cache_metadata, indent=2))
    print(f'Pulled and cached {len(documents):,} verified, non-excluded judgments: {CACHE_PATH}')

trial_rows: list[dict[str, Any]] = []
effect_rows: list[dict[str, Any]] = []
EFFECT_COLUMNS = [
    'case_id', 'trial_index', 'stage', 'original_factor', 'canonical_factor',
    'adjustment_months', 'base_months', 'effect_fraction', 'source', 'inferred',
]

for document in documents:
    judgement = document.get('judgement') or {}
    citation = judgement.get('neutral_citation') or document.get('filename')
    case_id = str(citation or document.get('source_judgement_id') or document['_id'])
    trials = (document.get('trials') or {}).get('trials') or []

    for trial_index, trial in enumerate(trials):
        charge = trial.get('charge_type') or {}
        drugs = trial.get('drugs') or []
        aggravating = trial.get('aggravating_factors') or []
        mitigating = trial.get('mitigating_factors') or []
        plea = trial.get('guilty_plea') or {}
        starting_detail = trial.get('starting_point')
        after_role_detail = trial.get('sentence_after_role')
        notional_detail = trial.get('notional_sentence')
        mitigation_detail = trial.get('mitigation_reduction')
        final_detail = trial.get('final_sentence')
        starting = total_months(starting_detail)
        after_role = total_months(after_role_detail)
        notional = total_months(notional_detail)
        mitigation_reduction = (
            float(mitigation_detail['reduction_months'])
            if mitigation_detail and mitigation_detail.get('reduction_months') is not None
            else None
        )
        pre_plea = notional - (mitigation_reduction or 0) if notional is not None else None
        final = total_months(final_detail)
        original_aggravating = [item.get('factor') for item in aggravating if item.get('factor')]
        original_mitigating = [item.get('factor') for item in mitigating if item.get('factor')]
        canonical_aggravating = unique_canonical_factors(aggravating)
        canonical_mitigating = unique_canonical_factors(mitigating)
        role_factors = [name for name in canonical_aggravating if name == 'Role of the defendant']
        other_aggravating = [name for name in canonical_aggravating if name != 'Role of the defendant']

        row = {
            'case_id': case_id,
            'neutral_citation': citation,
            'source_judgement_id': str(document.get('source_judgement_id') or ''),
            'filename': document.get('filename'),
            'trial_index': trial_index,
            'charge_no': charge.get('charge_no'),
            'defendant_id': charge.get('defendant_id'),
            'defendant_name': charge.get('defendant_name'),
            'drugs_json': json.dumps(drugs, default=str),
            'invalid_drug_quantity_count': 0,
            'invalid_drug_quantities': '',
            'aggravating_details_json': json.dumps(aggravating, default=str),
            'mitigating_details_json': json.dumps(mitigating, default=str),
            'guilty_plea_json': json.dumps(plea, default=str),
            'original_aggravating_factors': ' | '.join(original_aggravating),
            'original_mitigating_factors': ' | '.join(original_mitigating),
            'canonical_aggravating_factors': canonical_aggravating,
            'canonical_mitigating_factors': canonical_mitigating,
            'role_factors': role_factors,
            'other_aggravating_factors': other_aggravating,
            'starting_point_months': starting,
            'starting_point_inferred': is_inferred(starting_detail),
            'sentence_after_role_months': after_role,
            'sentence_after_role_inferred': is_inferred(after_role_detail),
            'notional_sentence_months': notional,
            'notional_sentence_inferred': is_inferred(notional_detail),
            'mitigation_reduction_months': mitigation_reduction,
            'mitigation_reduction_inferred': is_inferred(mitigation_detail),
            'pre_plea_months': pre_plea,
            'final_sentence_months': final,
            'final_sentence_inferred': is_inferred(final_detail),
        }
        for drug in drugs:
            drug_type = drug.get('drug_type')
            if not drug_type:
                continue
            raw_quantity = drug.get('quantity')
            try:
                quantity = float(raw_quantity)
            except (TypeError, ValueError):
                quantity = np.nan
            if not np.isfinite(quantity) or quantity < 0:
                row['invalid_drug_quantity_count'] += 1
                invalid_value = f'{drug_type}:{raw_quantity}'
                row['invalid_drug_quantities'] = ' | '.join(
                    filter(None, [row['invalid_drug_quantities'], invalid_value])
                )
                quantity = 0.0
            row[f'drug::{drug_type}'] = row.get(f'drug::{drug_type}', 0.0) + quantity
        trial_rows.append(row)

        for factor in aggravating:
            adjustment = factor.get('enhancement_months')
            canonical = canonical_factor(factor.get('factor'))
            stage = 'role' if canonical == 'Role of the defendant' else 'aggravation'
            base = starting if stage == 'role' else after_role
            if adjustment is not None and canonical and base and base > 0 and not is_inferred(factor):
                effect_rows.append({
                    'case_id': case_id,
                    'trial_index': trial_index,
                    'stage': stage,
                    'original_factor': factor.get('factor'),
                    'canonical_factor': canonical,
                    'adjustment_months': float(adjustment),
                    'base_months': float(base),
                    'effect_fraction': float(adjustment) / float(base),
                    'source': factor.get('source'),
                    'inferred': is_inferred(factor),
                })

        for factor in mitigating:
            adjustment = factor.get('reduction_months')
            canonical = canonical_factor(factor.get('factor'))
            if adjustment is not None and canonical and notional and notional > 0 and not is_inferred(factor):
                effect_rows.append({
                    'case_id': case_id,
                    'trial_index': trial_index,
                    'stage': 'mitigation',
                    'original_factor': factor.get('factor'),
                    'canonical_factor': canonical,
                    'adjustment_months': float(adjustment),
                    'base_months': float(notional),
                    'effect_fraction': float(adjustment) / float(notional),
                    'source': factor.get('source'),
                    'inferred': is_inferred(factor),
                })

        plea_adjustment = direct_plea_reduction(plea, pre_plea)
        plea_stage = plea.get('high_court_stage') or plea.get('district_court_stage') or 'Unknown'
        if plea_adjustment is not None and pre_plea and pre_plea > 0:
            effect_rows.append({
                'case_id': case_id,
                'trial_index': trial_index,
                'stage': 'plea',
                'original_factor': 'Guilty plea',
                'canonical_factor': f'Guilty plea: {plea_stage}',
                'adjustment_months': plea_adjustment,
                'base_months': float(pre_plea),
                'effect_fraction': plea_adjustment / float(pre_plea),
                'source': plea.get('source'),
                'inferred': False,
            })

trials_df = pd.DataFrame(trial_rows)
effects_df = pd.DataFrame(effect_rows, columns=EFFECT_COLUMNS)
if trials_df.empty:
    raise RuntimeError('No trial rows were found in verified-features')

assert canonical_factor('Import') == 'Cross-border trafficking'
assert canonical_factor('Export') == 'Cross-border trafficking'
assert canonical_factor('Refugee/Asylum') == 'Refugee claimant'
assert canonical_factor('Illegal immigrant') == 'Illegal immigrant'
assert unique_canonical_factors([{'factor': 'Import'}, {'factor': 'Export'}]) == ['Cross-border trafficking']

print(f'Flattened {len(trials_df):,} trials and {len(effects_df):,} direct factor adjustments')
trials_df.head()

Loaded 2,123 verified, non-excluded judgments from cache: /Users/cxiang/Projects/drug-trafficing-sentence-predictor/notebooks/.cache/stage_model_analysis_verified_features.json
Cache created: 2026-07-19T06:22:21.011966+00:00
Flattened 2,646 trials and 3,197 direct factor adjustments


,case_id,neutral_citation,source_judgement_id,filename,trial_index,charge_no,defendant_id,defendant_name,drugs_json,invalid_drug_quantity_count,invalid_drug_quantities,aggravating_details_json,mitigating_details_json,guilty_plea_json,original_aggravating_factors,original_mitigating_factors,canonical_aggravating_factors,canonical_mitigating_factors,role_factors,other_aggravating_factors,starting_point_months,starting_point_inferred,sentence_after_role_months,sentence_after_role_inferred,notional_sentence_months,notional_sentence_inferred,mitigation_reduction_months,mitigation_reduction_inferred,pre_plea_months,final_sentence_months,final_sentence_inferred,drug::Methamphetamine,drug::Cocaine,drug::Ketamine,drug::Ecstasy,drug::Nimetazepam,drug::Heroin,drug::Cannabis,drug::Fluorodeschloroketamine,drug::Other,drug::THC/CBD,drug::GHB/GBL,drug::Morphine,drug::Cathinones,drug::Etomidate
0,[2021] HKDC 461,[2021] HKDC 461,69b2f26d296bb506fb4884fc,None,0,1,1,WONG SZE TUNG,"[{""drug_type"": ""Methamphetamine"", ""other_drug_type"": null, ""quantity"": 5.52, ""source"": ""The particulars are that she, on 19 August 2020,...",0,,"[{""factor"": ""Persistent offender"", ""other_factor"": null, ""enhancement_months"": 4, ""inferred"": false, ""source"": ""Ms Wong has a bad record...","[{""factor"": ""Other"", ""other_factor"": ""Pregnant at time of offence / birth of baby; mercy to return to family"", ""reduction_months"": 2, ""r...","{""pleaded_guilty"": true, ""court_type"": ""District Court"", ""high_court_stage"": null, ""high_court_stage_other"": null, ""district_court_stage...",Persistent offender,Other,[Persistent offender],[Other],[],[Persistent offender],62.0,False,62.0,True,66.0,False,2.0,False,64.0,42.0,False,5.52,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,[2025] HKCFI 4288,[2025] HKCFI 4288,69b2f273296bb506fb488cd8,None,0,1,1,"Yim Kwok-yin, Alex (嚴國賢)","[{""drug_type"": ""Cocaine"", ""other_drug_type"": null, ""quantity"": 1637, ""source"": ""Upon search, two blocks of dangerous drugs, later confir...",0,,[],[],"{""pleaded_guilty"": true, ""court_type"": ""High Court"", ""high_court_stage"": ""Up to committal"", ""high_court_stage_other"": null, ""district_co...",,,[],[],[],[],241.0,True,241.0,True,241.0,True,NaN,False,241.0,160.0,False,NaN,1637.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,[2025] HKCFI 4288,[2025] HKCFI 4288,69b2f273296bb506fb488cd8,None,1,1,2,Lui Chi-man (呂志文),"[{""drug_type"": ""Cocaine"", ""other_drug_type"": null, ""quantity"": 1637, ""source"": ""Upon search, two blocks of dangerous drugs, later confir...",0,,[],"[{""factor"": ""Medical conditions"", ""other_factor"": null, ""reduction_months"": 2, ""reduction_percentage"": null, ""inferred"": false, ""source""...","{""pleaded_guilty"": true, ""court_type"": ""High Court"", ""high_court_stage"": ""Up to committal"", ""high_court_stage_other"": null, ""district_co...",,Medical conditions,[],[Medical conditions],[],[],241.0,False,241.0,True,241.0,False,2.0,False,239.0,158.0,False,NaN,1637.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,[2025] HKCFI 4288,[2025] HKCFI 4288,69b2f273296bb506fb488cd8,None,2,2,1,"Yim Kwok-yin, Alex (嚴國賢)","[{""drug_type"": ""Cocaine"", ""other_drug_type"": null, ""quantity"": 34292, ""source"": ""At around 3.49 pm, D1 was brought to the premises by th...",0,,[],[],"{""pleaded_guilty"": true, ""court_type"": ""High Court"", ""high_court_stage"": ""Up to committal"", ""high_court_stage_other"": null, ""district_co...",,,[],[],[],[],384.0,True,384.0,True,384.0,True,NaN,False,384.0,256.0,False,NaN,34292.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,[2021] HKCFI 1523,[2021] HKCFI 1523,69b2f26d296bb506fb4884f0,None,0,1,1,卓節巧,"[{""drug_type"": ""Methamphetamine"", ""other_drug_type"": null, ""quantity"": 25.28, ""source"": ""\u672c\u6848\u5605\u63a7\u7f6a\u6d89\u53ca\u560...",0,,[],"[{""factor"": ""Self-consumption"", ""other_factor"": ""Self-consumption of 20% of the drugs"", ""reduction_months"": 10, ""reduction_

## Fixed judgment-level train/test split

The split uses one row per judgment to ensure all charges from a case are in the same partition.

In [11]:
case_df = trials_df[['case_id', 'neutral_citation']].drop_duplicates().reset_index(drop=True)
if len(case_df) < 2:
    raise RuntimeError('At least two judgments are required for a train/test split')

splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_SEED)
train_case_indices, test_case_indices = next(splitter.split(case_df, groups=case_df['case_id']))
train_case_ids = set(case_df.iloc[train_case_indices]['case_id'])
test_case_ids = set(case_df.iloc[test_case_indices]['case_id'])

assert train_case_ids.isdisjoint(test_case_ids)
assert train_case_ids | test_case_ids == set(case_df['case_id'])

trials_df['partition'] = np.where(trials_df['case_id'].isin(test_case_ids), 'test', 'train')
effects_df['partition'] = np.where(effects_df['case_id'].isin(test_case_ids), 'test', 'train')
split_membership_df = case_df.assign(
    partition=np.where(case_df['case_id'].isin(test_case_ids), 'test', 'train'),
    random_seed=RANDOM_SEED,
    test_size=TEST_SIZE,
)

split_summary_df = pd.DataFrame([
    {'partition': 'train', 'judgments': len(train_case_ids), 'trials': int((trials_df['partition'] == 'train').sum())},
    {'partition': 'test', 'judgments': len(test_case_ids), 'trials': int((trials_df['partition'] == 'test').sum())},
])
split_summary_df

,partition,judgments,trials
0,train,1697,2127
1,test,425,519


## Train the starting-point model and learn direct factor effects

The starting-point regression is trained only on explicit starting points. Later-stage effects are judgment-level bootstrapped medians learned only from direct, non-inferred individual adjustments in the training partition.

In [12]:
drug_columns = sorted(column for column in trials_df.columns if column.startswith('drug::'))
if not drug_columns:
    raise RuntimeError('No drug quantities were found')
drug_feature_matrix = (
    trials_df[drug_columns]
    .apply(pd.to_numeric, errors='coerce')
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
    .clip(lower=0.0)
)
trials_df.loc[:, drug_columns] = drug_feature_matrix
assert np.isfinite(drug_feature_matrix.to_numpy(dtype=float)).all()
assert (drug_feature_matrix.to_numpy(dtype=float) >= 0).all()
invalid_drug_quantities_df = trials_df.loc[
    trials_df['invalid_drug_quantity_count'] > 0,
    ['case_id', 'neutral_citation', 'trial_index', 'charge_no', 'defendant_id',
     'invalid_drug_quantity_count', 'invalid_drug_quantities', 'drugs_json'],
].copy()
print(f"Invalid drug quantities replaced with zero for modelling: {len(invalid_drug_quantities_df):,}")

starting_train = trials_df.loc[
    (trials_df['partition'] == 'train')
    & trials_df['starting_point_months'].notna()
    & ~trials_df['starting_point_inferred']
].copy()
if len(starting_train) < 10:
    raise RuntimeError('Fewer than 10 explicit training starting points are available')

starting_model = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value=0.0)),
    ('log_quantity', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
    ('splines', SplineTransformer(n_knots=4, degree=2, extrapolation='linear')),
    ('scale', StandardScaler()),
    ('ridge', RidgeCV(alphas=np.logspace(-2, 4, 13))),
])
starting_model.fit(starting_train[drug_columns], starting_train['starting_point_months'])


def bootstrap_ci(values: pd.Series, seed_offset: int) -> tuple[float, float]:
    clean = values.dropna().to_numpy(dtype=float)
    if len(clean) == 0:
        return (np.nan, np.nan)
    generator = np.random.default_rng(RANDOM_SEED + seed_offset)
    medians = np.empty(BOOTSTRAP_ITERATIONS)
    for index in range(BOOTSTRAP_ITERATIONS):
        medians[index] = np.median(generator.choice(clean, size=len(clean), replace=True))
    return tuple(np.quantile(medians, [0.025, 0.975]).tolist())


def learn_effects(training_effects: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    grouped = training_effects.groupby(['stage', 'canonical_factor'], dropna=False)
    for seed_offset, ((stage, factor), group) in enumerate(grouped, start=1):
        by_case = group.groupby('case_id', as_index=False).agg(
            effect_fraction=('effect_fraction', 'median'),
            adjustment_months=('adjustment_months', 'median'),
            base_months=('base_months', 'median'),
        )
        support = len(group)
        case_support = len(by_case)
        effect_ci_low, effect_ci_high = bootstrap_ci(by_case['effect_fraction'], seed_offset)
        months_ci_low, months_ci_high = bootstrap_ci(by_case['adjustment_months'], seed_offset + 10_000)
        rows.append({
            'stage': stage,
            'canonical_factor': factor,
            'support_trials': support,
            'support_judgments': case_support,
            'supported': support >= MIN_FACTOR_SUPPORT,
            'median_effect_fraction': group['effect_fraction'].median(),
            'median_adjustment_months': group['adjustment_months'].median(),
            'median_base_months': group['base_months'].median(),
            'effect_ci_low': effect_ci_low,
            'effect_ci_high': effect_ci_high,
            'months_ci_low': months_ci_low,
            'months_ci_high': months_ci_high,
        })
    columns = [
        'stage', 'canonical_factor', 'support_trials', 'support_judgments', 'supported',
        'median_effect_fraction', 'median_adjustment_months', 'median_base_months',
        'effect_ci_low', 'effect_ci_high', 'months_ci_low', 'months_ci_high',
    ]
    return pd.DataFrame(rows, columns=columns).sort_values(['stage', 'canonical_factor']).reset_index(drop=True)


training_effects_df = effects_df.loc[effects_df['partition'] == 'train'].copy()
factor_effects_df = learn_effects(training_effects_df)
assert (factor_effects_df.loc[~factor_effects_df['supported'], 'support_trials'] < MIN_FACTOR_SUPPORT).all()
factor_effects_df

Invalid drug quantities replaced with zero for modelling: 1


,stage,canonical_factor,support_trials,support_judgments,supported,median_effect_fraction,median_adjustment_months,median_base_months,effect_ci_low,effect_ci_high,months_ci_low,months_ci_high
0,aggravation,Cross-border trafficking,126,117,True,0.058824,12.000000,228.000,0.055556,0.068182,11.000000,12.000000
1,aggravation,Illegal immigrant,4,4,False,0.080297,8.500000,106.000,0.000000,0.116279,0.000000,12.000000
2,aggravation,Multiple drugs,311,279,True,0.038462,3.000000,67.000,0.034483,0.040541,3.000000,3.000000
3,aggravation,On bail,43,40,True,0.042553,3.000000,72.000,0.033333,0.057112,3.000000,4.000000
4,aggravation,Other,70,61,True,0.062500,3.000000,60.250,0.054054,0.083333,3.000000,6.000000
5,aggravation,Persistent offender,213,199,True,0.038462,3.000000,87.900,0.033333,0.050000,3.000000,3.000000
6,aggravation,Refugee claimant,37,33,True,0.063745,6.000000,129.000,0.046512,0.082192,6.000000,8.000000
7,aggravation,Suspended sentence,4,4,False,0.051466,2.000000,58.500,0.014085,0.125000,0.500000,3.000000
8,aggravation,Use of minors,15,12,True,0.046154,5.000000,61.000,0.010875,0.099801,1.500000,6.000000
9,aggravation,Wanted,1,1,False,0.035714,3.000000,84.000,0.035714,0.035714,3.000000,3.000000


## Held-out stage prediction and metrics

In [13]:
supported_effects = factor_effects_df.loc[factor_effects_df['supported']].set_index(
    ['stage', 'canonical_factor']
)['median_effect_fraction'].to_dict()


def stage_effect(factors: list[str], stage: str, base_months: float) -> float:
    return sum(
        base_months * supported_effects.get((stage, factor), 0.0)
        for factor in dict.fromkeys(factors)
    )


def plea_factor(row: pd.Series) -> list[str]:
    plea = json.loads(row['guilty_plea_json'])
    if not plea.get('pleaded_guilty'):
        return []
    stage = plea.get('high_court_stage') or plea.get('district_court_stage') or 'Unknown'
    return [f'Guilty plea: {stage}']


test_df = trials_df.loc[trials_df['partition'] == 'test'].copy()
test_df['predicted_starting_point_months'] = np.clip(
    starting_model.predict(test_df[drug_columns]), 0, None
)
test_df['predicted_role_enhancement_months'] = test_df.apply(
    lambda row: stage_effect(row['role_factors'], 'role', row['predicted_starting_point_months']),
    axis=1,
)
test_df['predicted_sentence_after_role_months'] = np.clip(
    test_df['predicted_starting_point_months'] + test_df['predicted_role_enhancement_months'], 0, None
)
test_df['predicted_aggravation_months'] = test_df.apply(
    lambda row: stage_effect(
        row['other_aggravating_factors'], 'aggravation', row['predicted_sentence_after_role_months']
    ),
    axis=1,
)
test_df['predicted_notional_sentence_months'] = np.clip(
    test_df['predicted_sentence_after_role_months'] + test_df['predicted_aggravation_months'], 0, None
)
test_df['predicted_mitigation_reduction_months'] = test_df.apply(
    lambda row: np.clip(
        stage_effect(
            row['canonical_mitigating_factors'], 'mitigation', row['predicted_notional_sentence_months']
        ),
        0,
        row['predicted_notional_sentence_months'],
    ),
    axis=1,
)
test_df['predicted_pre_plea_months'] = np.clip(
    test_df['predicted_notional_sentence_months'] - test_df['predicted_mitigation_reduction_months'], 0, None
)
test_df['predicted_plea_reduction_months'] = test_df.apply(
    lambda row: np.clip(
        stage_effect(plea_factor(row), 'plea', row['predicted_pre_plea_months']),
        0,
        row['predicted_pre_plea_months'],
    ),
    axis=1,
)
test_df['predicted_final_sentence_months'] = np.clip(
    test_df['predicted_pre_plea_months'] - test_df['predicted_plea_reduction_months'], 0, None
)

assert (test_df.filter(regex='^predicted_').select_dtypes(include='number') >= 0).all().all()
assert np.allclose(
    test_df['predicted_final_sentence_months'],
    test_df['predicted_pre_plea_months'] - test_df['predicted_plea_reduction_months'],
)

metric_specs = [
    ('Starting point', 'starting_point_months', 'predicted_starting_point_months'),
    ('Sentence after role', 'sentence_after_role_months', 'predicted_sentence_after_role_months'),
    ('Notional sentence', 'notional_sentence_months', 'predicted_notional_sentence_months'),
    ('Non-plea mitigation reduction', 'mitigation_reduction_months', 'predicted_mitigation_reduction_months'),
    ('Pre-plea sentence', 'pre_plea_months', 'predicted_pre_plea_months'),
    ('Final sentence', 'final_sentence_months', 'predicted_final_sentence_months'),
]
metric_rows = []
for stage, actual_column, predicted_column in metric_specs:
    eligible = test_df[[actual_column, predicted_column]].dropna()
    if eligible.empty:
        metric_rows.append({'stage': stage, 'test_trials': 0, 'mae_months': np.nan, 'median_absolute_error_months': np.nan})
        continue
    metric_rows.append({
        'stage': stage,
        'test_trials': len(eligible),
        'mae_months': mean_absolute_error(eligible[actual_column], eligible[predicted_column]),
        'median_absolute_error_months': median_absolute_error(eligible[actual_column], eligible[predicted_column]),
    })
metrics_df = pd.DataFrame(metric_rows)

for _, actual_column, predicted_column in metric_specs:
    test_df[f'error::{actual_column}'] = test_df[predicted_column] - test_df[actual_column]

metrics_df

,stage,test_trials,mae_months,median_absolute_error_months
0,Starting point,473,13.997303,8.326851
1,Sentence after role,491,15.207375,8.326851
2,Notional sentence,519,15.265839,8.318089
3,Non-plea mitigation reduction,166,4.630316,1.824539
4,Pre-plea sentence,519,15.411949,8.637908
5,Final sentence,519,11.106788,6.165473


## Suspended sentence versus On bail

This report is a review aid only. The two factors stay separate unless the evidence criteria are met and a human explicitly approves their consolidation.

In [14]:
comparison_factors = ['Suspended sentence', 'On bail']
comparison_df = factor_effects_df.loc[
    (factor_effects_df['stage'] == 'aggravation')
    & factor_effects_df['canonical_factor'].isin(comparison_factors)
].copy()
median_reference = training_effects_df.loc[
    (training_effects_df['stage'] == 'aggravation')
    & training_effects_df['canonical_factor'].isin(comparison_factors),
    'base_months',
].median()
comparison_df['effect_at_median_reference_months'] = (
    comparison_df['median_effect_fraction'] * median_reference
)

merge_candidate = False
if set(comparison_df['canonical_factor']) == set(comparison_factors):
    suspended = comparison_df.set_index('canonical_factor').loc['Suspended sentence']
    on_bail = comparison_df.set_index('canonical_factor').loc['On bail']
    paired = training_effects_df.loc[
        (training_effects_df['stage'] == 'aggravation')
        & training_effects_df['canonical_factor'].isin(comparison_factors)
    ].groupby(['case_id', 'canonical_factor'])['effect_fraction'].median().unstack()
    difference_values = (paired.get('Suspended sentence') - paired.get('On bail')).dropna()
    difference_ci_low, difference_ci_high = bootstrap_ci(difference_values, 20_000)
    same_direction = np.sign(suspended['median_effect_fraction']) == np.sign(on_bail['median_effect_fraction'])
    practical_difference = abs(
        suspended['effect_at_median_reference_months'] - on_bail['effect_at_median_reference_months']
    )
    merge_candidate = bool(
        suspended['supported']
        and on_bail['supported']
        and same_direction
        and difference_ci_low <= 0 <= difference_ci_high
        and practical_difference <= 2
    )
    comparison_df['difference_effect_ci_low'] = difference_ci_low
    comparison_df['difference_effect_ci_high'] = difference_ci_high
    comparison_df['same_direction'] = same_direction
    comparison_df['practical_difference_months'] = practical_difference
comparison_df['merge_candidate_for_human_review'] = merge_candidate
comparison_df

,stage,canonical_factor,support_trials,support_judgments,supported,median_effect_fraction,median_adjustment_months,median_base_months,effect_ci_low,effect_ci_high,months_ci_low,months_ci_high,effect_at_median_reference_months,difference_effect_ci_low,difference_effect_ci_high,same_direction,practical_difference_months,merge_candidate_for_human_review
3,aggravation,On bail,43,40,True,0.042553,3.0,72.0,0.033333,0.057112,3.0,4.0,3.021277,NaN,NaN,True,0.632829,False
7,aggravation,Suspended sentence,4,4,False,0.051466,2.0,58.5,0.014085,0.125000,0.5,3.0,3.654106,NaN,NaN,True,0.632829,False


## Export reproducible analysis workbook

The export contains no MongoDB mutations. Re-reading the split sheet verifies that the exported partition membership matches the in-memory split.

In [15]:
EXCEL_ILLEGAL_CHARACTERS = re.compile(r'[\x00-\x08\x0B\x0C\x0E-\x1F]')


def excel_safe(value: Any) -> Any:
    if isinstance(value, str):
        return EXCEL_ILLEGAL_CHARACTERS.sub('', value)
    if isinstance(value, (list, tuple, dict)):
        return EXCEL_ILLEGAL_CHARACTERS.sub('', json.dumps(value, default=str))
    return value


def write_excel_sheet(writer: pd.ExcelWriter, dataframe: pd.DataFrame, sheet_name: str) -> None:
    dataframe.map(excel_safe).to_excel(writer, sheet_name=sheet_name, index=False)


export_trials_df = trials_df.copy()
for column in [
    'canonical_aggravating_factors',
    'canonical_mitigating_factors',
    'role_factors',
    'other_aggravating_factors',
]:
    export_trials_df[column] = export_trials_df[column].map(lambda value: ' | '.join(value))

with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
    write_excel_sheet(writer, split_summary_df, 'split summary')
    write_excel_sheet(writer, split_membership_df, 'split membership')
    write_excel_sheet(writer, export_trials_df, 'modelling rows')
    write_excel_sheet(writer, invalid_drug_quantities_df, 'invalid drug quantities')
    write_excel_sheet(writer, effects_df, 'direct adjustments')
    write_excel_sheet(writer, factor_effects_df, 'learned effects')
    write_excel_sheet(writer, comparison_df, 'bail vs suspended')
    write_excel_sheet(writer, metrics_df, 'held-out metrics')
    write_excel_sheet(writer, test_df, 'held-out predictions')

exported_split_df = pd.read_excel(OUTPUT_PATH, sheet_name='split membership')
expected_split = split_membership_df[['case_id', 'partition']].sort_values('case_id').reset_index(drop=True)
actual_split = exported_split_df[['case_id', 'partition']].sort_values('case_id').reset_index(drop=True)
pd.testing.assert_frame_equal(expected_split, actual_split)

print(f'Wrote {OUTPUT_PATH.resolve()}')
print('MongoDB was read only; no database writes were performed.')

Wrote /Users/cxiang/Projects/drug-trafficing-sentence-predictor/notebooks/stage_model_analysis.xlsx
MongoDB was read only; no database writes were performed.
